<a href="https://colab.research.google.com/github/Aldiss1/MPPM_Aldi/blob/main/JS09/JS09-Tugas1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install annoy faiss-cpu hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 8.2 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 19.6 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520035 sha256=ed0c7754519d38ce9f266f93eedc05c86782d21a5ff2f4f87828838aac7ad49a
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679795 sha256=838d65c836c06e0cfad7da13e0baa76da4efc0787cf023c8cdee541cf58e8403
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50

# Tugas Lab 1
Perbandingan Performansi ANNOY, FAISS, dan HNSW

1. Lakukan percobaan pada metric distance yang berbeda, 1000 vs 1.000.000 data, 2D vs 5D data, untuk algoritma,

   1. ANNOY
   2. FAISS
   3. HNSW

   Catat performansinya dalam bentuk tabel, misal

| Jumlah Data/Dimensi | ANNOY | FAISS | HNSW |
| ------------------- | ----- | ----- | ---- |
| 1000/2D             |       |       |      |
| 1000/5D             |       |       |      |
| 1000000/2D          |       |       |      |
| 1000000/5D          |       |       |      |

In [2]:
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
import pandas as pd

In [3]:
# -------------------------------
# Fungsi benchmark untuk tiap algoritma
# -------------------------------
k = 10

def run_annoy(X, query, metric='euclidean', n_trees=10, k=10):
    f = X.shape[1]
    index = AnnoyIndex(f, metric)
    for i, v in enumerate(X):
        index.add_item(i, v)
    start = time.time()
    index.build(n_trees)
    build_time = time.time() - start

    start = time.time()
    idx = index.get_nns_by_vector(query, k)
    query_time = time.time() - start
    return build_time, query_time, idx

def run_faiss(X, query, k=10):
    d = X.shape[1]
    index = faiss.IndexFlatL2(d)
    start = time.time()
    index.add(X)
    build_time = time.time() - start

    q = query.reshape(1, -1).astype('float32')
    start = time.time()
    D, I = index.search(q, k)
    query_time = time.time() - start
    return build_time, query_time, I[0]

def run_hnsw(X, query, ef=50, M=16, k=10):
    d = X.shape[1]
    n = X.shape[0]
    index = hnswlib.Index(space='l2', dim=d)
    start = time.time()
    index.init_index(max_elements=n, ef_construction=200, M=M)
    index.add_items(X)
    build_time = time.time() - start

    index.set_ef(ef)
    q = query.reshape(1, -1).astype('float32')
    start = time.time()
    labels, distances = index.knn_query(q, k=k)
    query_time = time.time() - start
    return build_time, query_time, labels[0]

In [4]:
# -------------------------------
# Jalankan percobaan untuk semua kombinasi
# -------------------------------
np.random.seed(42)
configs = [
    (1000, 2),
    (1000, 5),
    (1_000_000, 2),
    (1_000_000, 5),
]

rows = []
for n_data, dim in configs:
    X = np.random.random((n_data, dim)).astype(np.float32)
    query = np.random.random((dim,)).astype(np.float32)

    annoy_build, annoy_query, _ = run_annoy(X, query, k=k)
    faiss_build, faiss_query, _ = run_faiss(X, query, k=k)
    hnsw_build, hnsw_query, _ = run_hnsw(X, query, k=k)

    rows.append({
        "Jumlah Data/Dimensi": f"{n_data}/{dim}D",
        "ANNOY Build (s)": round(annoy_build, 4),
        "ANNOY Query (ms)": round(annoy_query * 1000, 4),
        "FAISS Build (s)": round(faiss_build, 4),
        "FAISS Query (ms)": round(faiss_query * 1000, 4),
        "HNSW Build (s)": round(hnsw_build, 4),
        "HNSW Query (ms)": round(hnsw_query * 1000, 4),
    })

df = pd.DataFrame(rows)
df

,Jumlah Data/Dimensi,ANNOY Build (s),ANNOY Query (ms),FAISS Build (s),FAISS Query (ms),HNSW Build (s),HNSW Query (ms)
0,1000/2D,0.0229,0.0596,0.0000,0.0663,0.0536,0.0494
1,1000/5D,0.0138,0.0532,0.0000,0.0350,0.0628,0.0539
2,1000000/2D,29.9499,0.0970,0.0019,9.7747,104.3660,0.0622
3,1000000/5D,17.8071,0.0823,0.0128,11.6620,158.7996,0.0770
